# データチェック
昨年度の坂本さん進捗分に対応したデータ確認➡CSV化。データは3回に分けて渡されてる。
- 20250825：初期サンプルデータ。
- 20251020：追加サンプルデータ。ヒーターに切り欠きがあったり、形状が異なったりするデータ
- 20251125：実製品形状のヒーター回路データ。これが一個しかない。

## 20250825
- NGKからの報告資料にあったやつ。
- 向こうのお仕事は：
    - 発熱と伝熱の解析を切り分けて、シミュレーション時間を減らせたこと
    - 発熱と伝熱の解析時のメッシュサイズを検討したこと。切り欠き(イベントって呼んでるね)とか厚み方向の数とか。
- こちらのお仕事：
  - 「発熱分布➡(伝熱)➡温度分布」のサロゲートモデル構築

↓もらったデータは、「発熱分布」と「温度分布」の一例。1回路のデータのみです。

In [16]:
import pandas as pd
from pathlib import Path

raw_path = Path("../data/raw/20250825_サンプルデータ/ヒータ発熱量.csv")
out_dir = Path("../data/processed/20250825_sample")
out_dir.mkdir(parents=True, exist_ok=True)

df_heater = pd.read_csv(raw_path, encoding="shift_jis")
df_heater.columns = df_heater.iloc[0]
df_heater = df_heater[1:].reset_index(drop=True)


def _object_flag(s: pd.Series) -> pd.Series:
    return s.map({"TRUE": 1, "FALSE": 0, True: 1, False: 0}).astype("Int64")


def _build(
    xyz: pd.DataFrame, values: pd.Series, obj: pd.Series, value_col: str
) -> pd.DataFrame:
    out = xyz.copy()
    out.columns = ["X", "Y", "Z"]
    out[value_col] = pd.to_numeric(values, errors="coerce")
    out["object_flag"] = _object_flag(obj)
    return out


xyz = df_heater.iloc[:, :3]
df_heater_out1 = _build(xyz, df_heater.iloc[:, 3], df_heater.iloc[:, 4], "heater_output_W_m-3")
df_heater_out2 = _build(xyz, df_heater.iloc[:, 5], df_heater.iloc[:, 6], "heater_output_W_m-3")

df_heater_out1.to_csv(out_dir / "heater_out_1.csv", index=False, encoding="utf-8")
df_heater_out2.to_csv(out_dir / "heater_out_2.csv", index=False, encoding="utf-8")

df_heater_out1.head()

,X,Y,Z,heater_output_W_m-3,object_flag
0,0,0,0.00141,NaN,0
1,0.0005,0,0.00141,NaN,0
2,0.001,0,0.00141,NaN,0
3,0.0015,0,0.00141,NaN,0
4,0.002,0,0.00141,NaN,0


In [17]:
raw_path = Path("../data/raw/20250825_サンプルデータ/セラ板上面温度.csv")
out_dir = Path("../data/processed/20250825_sample")
out_dir.mkdir(parents=True, exist_ok=True)

df_sera_temp = pd.read_csv(raw_path, encoding="shift_jis")
df_sera_temp.columns = df_sera_temp.iloc[0]
df_sera_temp = df_sera_temp[1:].reset_index(drop=True)

xyz = df_sera_temp.iloc[:, :3]
df_sera_temp_out1 = _build(xyz, df_sera_temp.iloc[:, 3], df_sera_temp.iloc[:, 4], "temperature_degC")
df_sera_temp_out2 = _build(xyz, df_sera_temp.iloc[:, 5], df_sera_temp.iloc[:, 6], "temperature_degC")

df_sera_temp_out1.to_csv(out_dir / "sera_temp_1.csv", index=False, encoding="utf-8")
df_sera_temp_out2.to_csv(out_dir / "sera_temp_2.csv", index=False, encoding="utf-8")

df_sera_temp_out1.head()

,X,Y,Z,temperature_degC,object_flag
0,0,0,0.003,64.779503,1
1,0.0005,0,0.003,64.810295,1
2,0.001,0,0.003,64.901070,1
3,0.0015,0,0.003,65.047394,1
4,0.002,0,0.003,65.241821,1


## 20251020
- 回路形状を変えた5パターンのデータを追加している。四角だったり丸だったり。
- 回路形状を変えたなかでも、発熱量が異なるデータを3条件おいてある。

In [19]:

raw_dir = Path("../data/raw/20251020_追加サンプルデータ/教師データ/ヒータ発熱量")
out_dir = Path("../data/processed/20251020_tuika_sample")
out_dir.mkdir(parents=True, exist_ok=True)

# 元CSV → 出力ファイル名のケースID（A/B/C = 1行目の 中 / 弱 / 強 の順）
CASE_BY_FILENAME = {
    "ヒータ発熱量_1_角型3mm.csv": "1",
    "ヒータ発熱量_1B_角型3mm切り欠き.csv": "1B",
    "ヒータ発熱量_2_円型3mm.csv": "2",
    "ヒータ発熱量_3_角型1.5mm.csv": "3",
    "ヒータ発熱量_4_円型1.5mm.csv": "4",
}
COND_SUFFIXES = ("A", "B", "C")

for fname, case_id in CASE_BY_FILENAME.items():
    df = pd.read_csv(raw_dir / fname, encoding="shift_jis")
    df.columns = df.iloc[0]
    df = df[1:].reset_index(drop=True)
    xyz = df.iloc[:, :3]
    for i, suf in enumerate(COND_SUFFIXES):
        j = 3 + i * 2
        out_df = _build(
            xyz, df.iloc[:, j], df.iloc[:, j + 1], "heater_output_W_m-3"
        )
        out_df.to_csv(
            out_dir / f"heater_out_{case_id}_{suf}.csv",
            index=False,
            encoding="utf-8",
        )

sorted(p.name for p in out_dir.glob("heater_out_*.csv"))

['heater_out_1B_A.csv',
 'heater_out_1B_B.csv',
 'heater_out_1B_C.csv',
 'heater_out_1_A.csv',
 'heater_out_1_B.csv',
 'heater_out_1_C.csv',
 'heater_out_2_A.csv',
 'heater_out_2_B.csv',
 'heater_out_2_C.csv',
 'heater_out_3_A.csv',
 'heater_out_3_B.csv',
 'heater_out_3_C.csv',
 'heater_out_4_A.csv',
 'heater_out_4_B.csv',
 'heater_out_4_C.csv']

In [20]:
# セル2を先に実行（_build, _object_flag）
raw_dir = Path("../data/raw/20251020_追加サンプルデータ/教師データ/セラ板上面温度")
out_dir = Path("../data/processed/20251020_tuika_sample")
out_dir.mkdir(parents=True, exist_ok=True)

CASE_BY_FILENAME = {
    "セラ板上面温度_1_角型3mm.csv": "1",
    "セラ板上面温度_1B_角型3mm切り欠き.csv": "1B",
    "セラ板上面温度_2_円型3mm.csv": "2",
    "セラ板上面温度_3_角型1.5mm.csv": "3",
    "セラ板上面温度_4_円型1.5mm.csv": "4",
}
COND_SUFFIXES = ("A", "B", "C")

for fname, case_id in CASE_BY_FILENAME.items():
    df = pd.read_csv(raw_dir / fname, encoding="shift_jis")
    df.columns = df.iloc[0]
    df = df[1:].reset_index(drop=True)
    xyz = df.iloc[:, :3]
    for i, suf in enumerate(COND_SUFFIXES):
        j = 3 + i * 2
        out_df = _build(
            xyz, df.iloc[:, j], df.iloc[:, j + 1], "temperature_degC"
        )
        out_df.to_csv(
            out_dir / f"sera_temp_{case_id}_{suf}.csv",
            index=False,
            encoding="utf-8",
        )

sorted(p.name for p in out_dir.glob("sera_temp_*.csv"))

['sera_temp_1B_A.csv',
 'sera_temp_1B_B.csv',
 'sera_temp_1B_C.csv',
 'sera_temp_1_A.csv',
 'sera_temp_1_B.csv',
 'sera_temp_1_C.csv',
 'sera_temp_2_A.csv',
 'sera_temp_2_B.csv',
 'sera_temp_2_C.csv',
 'sera_temp_3_A.csv',
 'sera_temp_3_B.csv',
 'sera_temp_3_C.csv',
 'sera_temp_4_A.csv',
 'sera_temp_4_B.csv',
 'sera_temp_4_C.csv']

## 20251125
- 実製品の形状でもやってみようとのことで、追加した1個だけのデータ

In [22]:
import pandas as pd
from pathlib import Path

# エクセル列: 1–2列目 XY、3列目 ヒータ熱流束、4列目 セラ板上面温度[K]、5列目 領域フラグ
# 各シートについて、熱流束用と温度用で5列CSVを2本ずつ保存する
xlsx_path = Path(
    "../data/raw/20251125_実製品形状データ/CAE-06-75-HPRE-D-2025-01_実製品形状データ.xlsx"
)
out_dir = Path("../data/processed/20251125_real_sample")
out_dir.mkdir(parents=True, exist_ok=True)


def _object_flag_from_col(s: pd.Series) -> pd.Series:
    return s.map({True: 1, False: 0, "TRUE": 1, "FALSE": 0}).astype("Int64")


with pd.ExcelFile(xlsx_path, engine="openpyxl") as xf:
    for sheet_idx, sheet_name in enumerate(xf.sheet_names):
        df = pd.read_excel(xf, sheet_name=sheet_idx, header=0)
        n = len(df)
        x = pd.to_numeric(df.iloc[:, 0], errors="coerce")
        y = pd.to_numeric(df.iloc[:, 1], errors="coerce")
        z = pd.Series([float("nan")] * n)
        heater_flux = pd.to_numeric(df.iloc[:, 2], errors="coerce")
        temp_k = pd.to_numeric(df.iloc[:, 3], errors="coerce")
        obj = _object_flag_from_col(df.iloc[:, 4])

        tag = sheet_idx + 1  # シート1, シート2, …
        df_h = pd.DataFrame(
            {
                "X": x,
                "Y": y,
                "Z": z,
                "heater_flux_W_mm-2": heater_flux,
                "object_flag": obj,
            }
        )
        df_t = pd.DataFrame(
            {
                "X": x,
                "Y": y,
                "Z": z,
                "temperature_K": temp_k - 273.15,
                "object_flag": obj,
            }
        )
        df_h.to_csv(
            out_dir / f"shape{tag}_heater_flux.csv",
            index=False,
            encoding="utf-8",
        )
        df_t.to_csv(
            out_dir / f"shape{tag}_temperature.csv",
            index=False,
            encoding="utf-8",
        )

sorted(p.name for p in out_dir.glob("shape*.csv"))

['shape1_heater_flux.csv',
 'shape1_temperature.csv',
 'shape2_heater_flux.csv',
 'shape2_temperature.csv']

## まとめ
- これらのデータを使ってモデル構築するには：
  - npzファイルに変換する。配列にする
  - NPZの可視化もしておきたい。
  - パッチの切り出し：予測に使う領域など、領域を区切る。
  - データおーぐめんてーしょん：画像を回転させたりして、データのかさましをする。